# `ov.synbio` — synthetic biology in OmicVerse

A three-layer design stack — **metabolism (A)**, **protein/enzyme (B)**, **DNA (C)** — with an A↔B hinge that lets an enzyme edit re-solve a metabolic yield.

Install: `pip install "omicverse[synbio]"`. GPU is used automatically for the protein models (ESMFold / ESM-2); everything else is CPU.

Discovery: every function is registered under the `synthetic_biology` category with Chinese + English aliases.

In [ ]:
import omicverse as ov
ov.find_function("酶")                              # fuzzy search across aliases
ov.list_functions(category="synthetic_biology")   # all synbio functions

## Layer A — genome-scale metabolic models (COBRApy)

In [ ]:
m = ov.synbio.load_gem("e_coli_core")   # BiGG id or local SBML
sol = ov.synbio.fba(m)
print("max growth:", round(sol.objective_value, 4), "/h")

In [ ]:
# strain design for a target product: amplification + knockout targets
res = ov.synbio.strain_design(m, "EX_succ_e")
print("amplify (FSEOF):", list(res.amplify['reaction'].head(5)))
res.knockout.head()

## Layer C — DNA design (DNAchisel / primer3)

In [ ]:
opt = ov.synbio.codon_optimize("MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQ", host="e_coli")
print(opt)
primers = ov.synbio.design_primers(opt.sequence * 3)
primers[0]

## Layer B — proteins & enzymes (GPU)

ESM-2 embeddings, zero-shot variant effects (in-silico directed evolution), ESMFold structure, ProteinMPNN inverse design & ΔΔG stability. Each prints the device it used.

In [ ]:
seq = "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQAPILSRVGDGTQDNLSGAEKAVQVKVKALPDAQFEVVHSLAKWKR"
X = ov.synbio.protein_embed([seq], model="esm2_t33_650M")   # (1, 1280)
X.shape

In [ ]:
# zero-shot saturation mutagenesis: which substitutions the model favours
dms = ov.synbio.variant_effect(seq[:50], model="esm1v")
dms.head()

In [ ]:
# ESMFold structure (GPU) -> PDB, then ProteinMPNN inverse design + ΔΔG
pred = ov.synbio.predict_structure(seq[:50], out_path="demo.pdb")
print("mean pLDDT:", round(pred.mean_plddt, 1), "| device:", pred.device)
designs = ov.synbio.inverse_design("demo.pdb", num_sequences=4)
designs[:2]

## The moat — A↔B coupling

`enzyme_kcat` (protein layer) → `ec_model` (metabolic layer) → `fba`: **edit the enzyme, re-solve the yield.** Under a fixed protein budget a faster enzyme variant relaxes the enzyme-capacity constraint and raises attainable growth.

In [ ]:
k = ov.synbio.enzyme_kcat(seq, "OCC1OC(O)(COP(=O)(O)O)C(O)C1O")   # enzyme + substrate SMILES
ecm = ov.synbio.ec_model(m, {"PFK": k.kcat})
pool = ecm.synbio_ec["total_protein"]
print("kcat =", round(k.kcat, 2), "/s  -> growth =", round(ov.synbio.fba(ecm).objective_value, 4))

In [ ]:
# same protein budget, faster enzyme variant -> higher yield
for factor in [0.3, 1.0, 3.0, 10.0]:
    ecm_v = ov.synbio.ec_model(m, {"PFK": k.kcat * factor}, total_protein=pool)
    g = ov.synbio.fba(ecm_v).objective_value
    print(f"kcat x{factor:>4}: growth = {g:.4f} /h")